# BME 574 — NumPy in one sitting
### Everything you need for Project 1, and nothing you don't

**Who this is for.** You have written some Python, or maybe none. You have not used NumPy. You
have two weeks and a project to do. This notebook is the shortest path from here to there.

**How to use it.** Run every cell in order. Each section ends with a short exercise and a
`check()` that tells you whether you got it right. Do not skip the exercises — reading NumPy and
writing NumPy are different skills, and the project needs the second one.

**Time.** About 40 minutes if you do the exercises. It will save you several days.

**The eleven things.** That is genuinely all Project 1 uses:

| | Operation | Why the project needs it |
|---|---|---|
| 1 | `np.array`, `.shape`, `.dtype` | Knowing what you are holding |
| 2 | Slicing `a[2:5]` | Taking pieces of data |
| 3 | Boolean masks `a[m]` | Train/test splits, selecting one class |
| 4 | Fancy indexing `a[idx]` | Applying a saved split |
| 5 | `.reshape` | Image ↔ row vector |
| 6 | `.T` (transpose) | The single most common bug in this course |
| 7 | `axis=` in `.mean`, `.sum` | Averaging over samples vs. over features |
| 8 | Broadcasting | Centering a matrix, in one line |
| 9 | `@` (matrix multiply) | Projecting onto components |
| 10 | `argsort`, `argmin` | Ranking, and nearest-centroid classification |
| 11 | `np.linalg.svd` | The whole point |

A twelfth thing, `matplotlib`, gets a short section at the end.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

PASSED = []

def check(name, condition, hint=''):
    """Tiny self-test helper used throughout this notebook."""
    if condition:
        PASSED.append(name)
        print(f'  correct  -  {name}')
    else:
        print(f'  not yet  -  {name}')
        if hint:
            print(f'            hint: {hint}')

print('ready — numpy', np.__version__)

ready — numpy 2.5.2


---
## 1 · An array is a grid of numbers that knows its own shape

A Python list is a bag of anything. A NumPy array is a rectangular grid of *numbers*, all the
same type, and it knows its dimensions. That is what makes it fast and what makes maths on it
readable.

`.shape` is a tuple: `(rows, columns)` for a 2-D array. **Print it constantly.** Ninety per cent
of NumPy bugs are shape bugs, and they are invisible until you look.

In [2]:
a = np.array([1, 2, 3, 4])
M = np.array([[1, 2, 3],
              [4, 5, 6]])

print('a       ', a)
print('a.shape ', a.shape)     # (4,)    one dimension, four elements
print()
print('M\n', M)
print('M.shape ', M.shape)     # (2, 3)  two rows, three columns
print('M.dtype ', M.dtype)     # int64 — whole numbers
print()
print('zeros(2,3)\n', np.zeros((2, 3)))
print('arange(5) ', np.arange(5))          # 0,1,2,3,4  — like range()
print('linspace  ', np.linspace(0, 1, 5))  # 5 evenly spaced values from 0 to 1

a        [1 2 3 4]
a.shape  (4,)

M
 [[1 2 3]
 [4 5 6]]
M.shape  (2, 3)
M.dtype  int64

zeros(2,3)
 [[0. 0. 0.]
 [0. 0. 0.]]
arange(5)  [0 1 2 3 4]
linspace   [0.   0.25 0.5  0.75 1.  ]


### A warning you will need later

`(4,)` and `(4, 1)` and `(1, 4)` are **three different shapes**. They look the same when printed
and they behave differently in arithmetic. This causes real confusion, so meet it now.

In [3]:
v = np.array([1, 2, 3, 4])
print('v.shape        ', v.shape)              # (4,)   a 1-D array
print('column shape   ', v.reshape(4, 1).shape) # (4,1)  a column
print('row shape      ', v.reshape(1, 4).shape) # (1,4)  a row
print()
print('MedMNIST hands you labels shaped (N, 1) rather than (N,).')
print('.ravel() flattens to 1-D and is how you fix that:')
labels_2d = np.array([[0], [1], [1], [0]])
print('  before', labels_2d.shape, ' after', labels_2d.ravel().shape)

v.shape         (4,)
column shape    (4, 1)
row shape       (1, 4)

MedMNIST hands you labels shaped (N, 1) rather than (N,).
.ravel() flattens to 1-D and is how you fix that:
  before (4, 1)  after (4,)


**Exercise 1.** Make a 3×4 array of zeros and store it in `E1`.

In [4]:
E1 = ...   # your code here

check('E1 is a 3x4 array of zeros',
      isinstance(E1, np.ndarray) and E1.shape == (3, 4) and (E1 == 0).all(),
      'np.zeros takes a SHAPE TUPLE: np.zeros((rows, cols))')

  not yet  -  E1 is a 3x4 array of zeros
            hint: np.zeros takes a SHAPE TUPLE: np.zeros((rows, cols))


---
## 2 · Slicing: taking pieces

`start:stop` — includes `start`, excludes `stop`. For 2-D arrays you give **two** slices
separated by a comma: `M[rows, columns]`. A bare `:` means "all of them".

In [ ]:
M = np.arange(12).reshape(3, 4)
print('M\n', M)
print()
print('M[0]        row 0        ', M[0])
print('M[:, 0]     column 0     ', M[:, 0])
print('M[1, 2]     one element  ', M[1, 2])
print('M[0:2, 1:3] a block\n', M[0:2, 1:3])
print()
print('M[-1]       last row     ', M[-1])
print('M[:, :2]    first 2 cols\n', M[:, :2])

**Exercise 2.** From `M` above, take the last two columns into `E2`. Expected shape `(3, 2)`.

In [ ]:
M = np.arange(12).reshape(3, 4)
E2 = ...   # your code here

check('E2 is the last two columns',
      isinstance(E2, np.ndarray) and E2.shape == (3, 2) and (E2 == M[:, 2:]).all(),
      'you want ALL rows and the last two columns: M[:, 2:]')

---
## 3 · Boolean masks — how you split data

Compare an array to something and you get an array of `True`/`False` the same shape. Use that as
an index and you get back only the elements where it was `True`.

**This is how train/test splits work**, so it is worth being comfortable here.

In [ ]:
x = np.array([10, 20, 30, 40, 50])
mask = x > 25
print('x     ', x)
print('x > 25', mask)
print('x[mask]', x[mask])
print()

# the realistic version: rows of a data matrix belonging to one subject
subject = np.array([0, 0, 0, 1, 1, 2, 2, 2, 2])
X = np.arange(9 * 2).reshape(9, 2)          # 9 observations, 2 features

is_subj1 = subject == 1
print('rows from subject 1:\n', X[is_subj1])
print()
print('count per subject:', np.bincount(subject))
print('how many rows are subject 0 or 2:', np.isin(subject, [0, 2]).sum())

**Exercise 3.** Using `subject` above, select the rows of `X` that come from subject 0 **or**
subject 2, into `E3`. Expected shape `(7, 2)`.

In [ ]:
subject = np.array([0, 0, 0, 1, 1, 2, 2, 2, 2])
X = np.arange(9 * 2).reshape(9, 2)
E3 = ...   # your code here

check('E3 has the rows from subjects 0 and 2',
      isinstance(E3, np.ndarray) and E3.shape == (7, 2)
      and (E3 == X[np.isin(subject, [0, 2])]).all(),
      'np.isin(subject, [0, 2]) gives you the mask; use it to index X')

---
## 4 · Fancy indexing — applying a saved split

You can also index with a list of positions. This is how you apply a split that someone else
(me) has already computed and handed you in a file.

In [ ]:
X = np.arange(9 * 2).reshape(9, 2)

train_idx = np.array([0, 1, 2, 5, 6, 7])
test_idx = np.array([3, 4, 8])

X_train = X[train_idx]
X_test = X[test_idx]
print('X_train.shape', X_train.shape)
print('X_test.shape ', X_test.shape)
print()

# always worth asserting — this catches a whole class of silent bugs
overlap = np.intersect1d(train_idx, test_idx)
print('overlap between train and test:', overlap, '(should be empty)')
assert len(overlap) == 0

---
## 5 & 6 · reshape and transpose — where the project actually breaks

This section matters more than the rest combined.

An image is a 2-D grid. A data matrix wants each image to be **one long row**. `reshape` moves
between those two views without changing any numbers — it only changes how they are read.

`-1` means "work it out for me".

In [ ]:
img = np.arange(12).reshape(3, 4)      # a tiny 3x4 'image'
print('img\n', img)
print()

row = img.reshape(-1)                  # flatten to one long row
print('flattened      ', row, row.shape)
print('back to image\n', row.reshape(3, 4))
print()

# a STACK of images -> a data matrix
stack = np.arange(5 * 3 * 4).reshape(5, 3, 4)   # 5 images, each 3x4
Xmat = stack.reshape(5, -1)                      # 5 rows, 12 columns
print('stack.shape', stack.shape, ' ->  Xmat.shape', Xmat.shape)
print('one row is one whole image:', Xmat[0])

### Transpose

`.T` flips rows and columns. It is one character, it never errors, and **it is the most common
bug in this course** — because code with a wrong transpose still runs and still produces
plausible-looking pictures.

The class face data stores images as *columns*. Almost everything else, including scikit-learn,
wants observations as *rows*. Transpose once, deliberately, and print the shape.

In [ ]:
M = np.arange(6).reshape(2, 3)
print('M   ', M.shape, '\n', M)
print('M.T ', M.T.shape, '\n', M.T)
print()

faces_like = np.arange(12 * 5).reshape(12, 5)   # 12 pixels x 5 images (column-major)
X = faces_like.T                                 # 5 images x 12 pixels (row-major)
print('stored as ', faces_like.shape, ' = pixels x images')
print('we want   ', X.shape, ' = images x pixels')
print()
print('Rule for this course: rows are observations. Always.')

**Exercise 5.** You are handed `stack`, 8 images each 5×6. Turn it into a data matrix `E5`
with one row per image.

In [ ]:
stack = np.arange(8 * 5 * 6).reshape(8, 5, 6)
E5 = ...   # your code here

check('E5 is 8 rows of 30 pixels',
      isinstance(E5, np.ndarray) and E5.shape == (8, 30)
      and (E5[0] == stack[0].reshape(-1)).all(),
      'stack.reshape(8, -1)  — keep the first axis, flatten the rest')

---
## 7 · `axis=` — averaging over the right thing

For a matrix where **rows are observations and columns are features**:

- `axis=0` collapses **down** the rows → one number per **feature**. This is the mean face.
- `axis=1` collapses **across** the columns → one number per **observation**.

Getting this backwards is a silent error: both give you an array, neither complains.

The trick that makes it stick: **`axis=` names the axis that disappears.**

In [ ]:
X = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.],
              [10., 11., 12.]])      # 4 observations, 3 features
print('X.shape', X.shape)
print()
print('X.mean(axis=0)', X.mean(axis=0), ' <- one per FEATURE  (the mean observation)')
print('X.mean(axis=1)', X.mean(axis=1), ' <- one per OBSERVATION')
print()
print('shapes:', X.mean(axis=0).shape, 'and', X.mean(axis=1).shape)
print()
print('also: .sum, .std, .min, .max, .argmax all take axis=')
print('X.std(axis=0) ', X.std(axis=0).round(2))

**Exercise 7.** Compute the mean *feature values* of `X` (the 'mean observation') into `E7`.

In [ ]:
X = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.], [10., 11., 12.]])
E7 = ...   # your code here

check('E7 is the per-feature mean',
      isinstance(E7, np.ndarray) and E7.shape == (3,) and np.allclose(E7, [5.5, 6.5, 7.5]),
      'you want one number per column, so the ROW axis disappears: axis=0')

---
## 8 · Broadcasting — centering in one line

If you subtract a `(3,)` array from a `(4, 3)` array, NumPy quietly repeats the small one for
every row. That is broadcasting, and it is how you centre a data matrix without a loop.

Centering — subtracting the mean of each feature — is what turns an SVD into a PCA. You will do
it in the project.

In [ ]:
X = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.], [10., 11., 12.]])

mu = X.mean(axis=0)          # (3,)
Xc = X - mu                  # (4,3) - (3,)  ->  broadcast down the rows

print('mu       ', mu)
print('centred\n', Xc)
print()
print('each column of the centred matrix now averages to 0:', Xc.mean(axis=0).round(10))
print()
print('This is the whole of "centre the data". No loop needed.')

> **The project rule.** `mu` must be computed from the **training rows only**, then subtracted
> from both training and test. Computing it from everything is data leakage — the single most
> important idea in Project 1.

In [ ]:
# what that looks like in practice
train_idx = np.array([0, 1, 2])
test_idx = np.array([3])

mu = X[train_idx].mean(axis=0)      # <- training rows only
Xc_train = X[train_idx] - mu
Xc_test = X[test_idx] - mu          # <- same mu
print('mu from training rows only:', mu)

---
## 9 · `@` — matrix multiplication, which is how you project

`*` multiplies element by element. `@` does real matrix multiplication. You want `@`.

The shape rule: `(a, b) @ (b, c)` gives `(a, c)`. **The inner numbers must match**, and if they
do not, NumPy raises an error — which is a kindness, because it catches your transpose bug.

In [ ]:
A = np.arange(6).reshape(2, 3)     # (2,3)
B = np.arange(12).reshape(3, 4)    # (3,4)
print('A @ B shape:', (A @ B).shape, ' <- (2,3) @ (3,4) = (2,4)')
print()

# a deliberate mismatch, so you recognise the error when you cause one
try:
    B @ A                              # (3,4) @ (2,3) — inner numbers disagree
except ValueError as e:
    print('mismatched shapes raise:')
    print('   ', str(e)[:90])
print()
print('* is elementwise and is NOT what you want here:')
print((np.ones((2, 3)) * np.ones((2, 3))).shape)

### The projection you will write in the project

If `Vt` holds the components as rows — shape `(k, n_features)` — then projecting your centred
data onto them is one line:

```python
Z = (X - mu) @ Vt[:k].T
```

Check the shapes: `(n_samples, n_features) @ (n_features, k)` → `(n_samples, k)`. Each row is one
observation described by `k` numbers instead of thousands. That is the whole idea of the course
so far.

In [ ]:
n_samples, n_features, k = 20, 50, 4
X = np.random.default_rng(0).normal(size=(n_samples, n_features))
mu = X.mean(axis=0)
Vt = np.linalg.svd(X - mu, full_matrices=False)[2]

Z = (X - mu) @ Vt[:k].T
print('X   ', X.shape)
print('Vt  ', Vt.shape, ' -> Vt[:k].T is', Vt[:k].T.shape)
print('Z   ', Z.shape, ' <- each observation is now', k, 'numbers')

---
## 10 · `argsort` and `argmin` — ranking, and classifying

`sort` gives you the sorted values. `argsort` gives you the **positions** that would sort them,
which is what you actually want when you need to know *which* item won.

In [ ]:
scores = np.array([0.3, 0.9, 0.1, 0.7])
print('scores        ', scores)
print('argsort       ', np.argsort(scores), ' <- positions, smallest first')
print('argsort[::-1] ', np.argsort(scores)[::-1], ' <- largest first')
print('argmax        ', np.argmax(scores), ' <- position of the biggest')
print('argmin        ', np.argmin(scores))
print()
print('top 2 positions:', np.argsort(scores)[::-1][:2])

### Nearest-centroid classification, in full

This is the entire classifier the project asks for. It is the same method we used to recognize
faces: find the average of each class, then assign a new point to whichever average is closest.

In [ ]:
rng = np.random.default_rng(0)
Z_train = np.vstack([rng.normal(0, 1, (20, 2)), rng.normal(4, 1, (20, 2))])
y_train = np.array([0] * 20 + [1] * 20)
Z_test = np.array([[0.5, 0.2], [3.8, 4.1], [2.0, 2.0]])

# 1. the average position of each class
classes = np.unique(y_train)
centroids = np.stack([Z_train[y_train == c].mean(axis=0) for c in classes])
print('centroids\n', centroids.round(2))
print()

# 2. distance from every test point to every centroid
d = ((Z_test[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=-1)
print('distances (rows = test points, cols = classes)\n', d.round(2))
print()

# 3. the closest one wins
pred = classes[np.argmin(d, axis=1)]
print('predictions', pred)
print()
print('accuracy against known answers:', (pred == np.array([0, 1, 0])).mean())

> The `[:, None, :]` trick inserts a length-1 axis so broadcasting compares every test point
> against every centroid at once. You do not have to love it — you do have to recognize it. If it
> bothers you, a plain loop over centroids is equally correct and easier to read.

---
## 11 · `np.linalg.svd` — what actually comes back

One line, three outputs. The thing that confuses people is `Vt`: it is **V transposed**, so the
components are its *rows*, not its columns.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(30, 8))
mu = X.mean(axis=0)
Xc = X - mu

U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
print('Xc', Xc.shape)
print('U ', U.shape, ' <- one row per observation')
print('s ', s.shape, ' <- singular values, largest first, ALWAYS 1-D')
print('Vt', Vt.shape, ' <- one row per COMPONENT')
print()
print('component 1 is Vt[0], not Vt[:, 0].  This trips up everyone once.')
print()
print('s (first 4):', s[:4].round(3))
print('variance explained by the first 3:',
      round(float((s[:3] ** 2).sum() / (s ** 2).sum()), 3))
print()
print('rebuild the original from the pieces:')
print('  max error', np.abs(Xc - (U * s) @ Vt).max())
print()
print('always pass full_matrices=False — otherwise U is enormous and you do not need it.')

**Exercise 11.** Compute the fraction of variance explained by the first **two** components of
`s` into `E11`. (Variance goes with the *square* of a singular value.)

In [ ]:
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
E11 = ...   # your code here — a single number between 0 and 1

check('E11 is the variance explained by 2 components',
      np.isscalar(E11) or (isinstance(E11, (np.floating, float))),
      'square the singular values, sum the first two, divide by the total')
target = float((s[:2] ** 2).sum() / (s ** 2).sum())
check('E11 has the right value',
      isinstance(E11, (float, np.floating)) and np.isclose(float(E11), target),
      '(s[:2]**2).sum() / (s**2).sum()')

---
## 12 · Plotting — the four calls you need

`plot` for curves, `semilogy` for singular values, `scatter` for projections, `imshow` for
anything image-shaped. Label your axes; an unlabelled figure loses marks and, more importantly,
cannot be understood six weeks later.

In [ ]:
rng = np.random.default_rng(1)
fig, ax = plt.subplots(2, 2, figsize=(9, 6.5))

ax[0, 0].plot(np.linspace(0, 4 * np.pi, 200), np.sin(np.linspace(0, 4 * np.pi, 200)))
ax[0, 0].set(xlabel='time', ylabel='amplitude', title='plot — a curve')

ax[0, 1].semilogy(np.exp(-np.arange(30) / 5) + 1e-3, 'o-', ms=3)
ax[0, 1].set(xlabel='index', ylabel='singular value', title='semilogy — a spectrum')

g1, g2 = rng.normal(0, 1, (40, 2)), rng.normal(3, 1, (40, 2))
ax[1, 0].scatter(g1[:, 0], g1[:, 1], s=18, label='class 0')
ax[1, 0].scatter(g2[:, 0], g2[:, 1], s=18, label='class 1')
ax[1, 0].set(xlabel='PC 1', ylabel='PC 2', title='scatter — a projection')
ax[1, 0].legend(fontsize=8)

ax[1, 1].imshow(rng.normal(size=(20, 20)), cmap='gray')
ax[1, 1].set_title('imshow — anything image-shaped')
ax[1, 1].axis('off')

fig.tight_layout(); plt.show()

---
## The five habits that will save you

1. **Print `.shape` after every reshape, transpose, and slice.** Every single one. It costs a
   second and it is how you find the bug before it becomes a wrong figure.
2. **Assert what you believe.** `assert X.shape == (n, p)` turns a silent wrong answer into a
   loud stop.
3. **`axis=` names the axis that disappears.** Say it out loud once and you will stop guessing.
4. **`Vt[0]` is the first component**, not `Vt[:, 0]`.
5. **Anything fitted from data — a mean, a scaler, an SVD basis — is fitted on training rows
   only.** This is the idea Project 1 is really about.

### Where to look things up

The NumPy docs are good and searchable: `numpy.org/doc/stable`. For anything in this notebook,
`help(np.mean)` in a cell works offline. AI assistants are allowed and encouraged — subject to
the course policy: disclose, verify, understand. If an assistant hands you a line you cannot
explain, that line is not ready to submit.

In [ ]:
print(f'exercises passed: {len(set(PASSED))}')
for name in dict.fromkeys(PASSED):
    print('  -', name)
print()
if len(set(PASSED)) >= 6:
    print('You have what Project 1 needs. Open the starter notebook.')
else:
    print('Go back and finish the exercises marked "not yet" before starting the project.')